In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import numpy as np
import re
import wandb

In [3]:
from datasets import load_dataset

# Load the math dataset from Hugging Face
dataset = load_dataset("lighteval/MATH", split="test",trust_remote_code=True)
print(dataset)
print(dataset[0])
for i in range(10):
    print(len(dataset[i]['problem']))

dataset = load_dataset("lighteval/MATH", split="all",trust_remote_code=True)
print(dataset)
print(dataset[0])
for i in range(7500,7510):
    print(len(dataset[i]['problem']))   


Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 5000
})
{'problem': 'How many vertical asymptotes does the graph of $y=\\frac{2}{x^2+x-6}$ have?', 'level': 'Level 3', 'type': 'Algebra', 'solution': 'The denominator of the rational function factors into $x^2+x-6=(x-2)(x+3)$. Since the numerator is always nonzero, there is a vertical asymptote whenever the denominator is $0$, which occurs for $x = 2$ and $x = -3$.  Therefore, the graph has $\\boxed{2}$ vertical asymptotes.'}
74
72
91
30
39
69
103
253
240
108
Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 12500
})
{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solutio

In [4]:
# Load the LLaMA 3 8B model and tokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
device = torch.device("cuda:0") 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_cache=False,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    device_map = device,
)
model.config.use_flash_attention = True

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left' # to prevent errors with FA
tokenizer.truncation_side = 'left' # to prevent cutting off last generation


def select_questions_batch(dataset, indices):
    """Selects a batch of questions from the dataset by indices."""
    questions = [dataset[i]["problem"] for i in indices]
    solutions = [dataset[i]["solution"] for i in indices]
    return questions, solutions

def generate_batch(model, tokenizer, input_texts, number_of_samples, max_new_tokens):
    """Generates responses for a batch of input texts."""
    inputs = tokenizer(input_texts, return_tensors="pt", truncation=True, padding=True, max_length=2048).to(device)
    log_all_token_probs = torch.zeros([len(input_texts), number_of_samples,max_new_tokens]) + 100
    print(f'running model.generate for {len(input_texts)} inputs')
    output_ = model.generate(
        **inputs,
        do_sample=True,
        max_new_tokens=max_new_tokens,
        temperature=0.9,
        num_return_sequences=number_of_samples,
        return_dict_in_generate=True,
        output_scores=True,
        use_cache=True,
    )
    print(f'finished model.generate for {len(input_texts)} inputs')
    batch_solutions = []
    batch_probabilities = []
    
    for j, input_text in enumerate(input_texts):  # Process each input in the batch
        solutions = []
        probabilities = []
        for i in range(number_of_samples):
            idx = j * number_of_samples + i
            output = output_.sequences[idx].tolist()[len(inputs["input_ids"][j]):]
            # Find the position of the EOS token and truncate if it exists
            if tokenizer.eos_token_id in output:
                eos_position = output.index(tokenizer.eos_token_id)
                output = output[:eos_position]
            response = tokenizer.decode(output)
            
            scores = output_.scores

            log_token_probs = [score[idx].log_softmax(dim=-1)[token].item() for score, token in zip(scores, output)]
            log_all_token_probs[j, i, :len(log_token_probs)] = torch.tensor(log_token_probs)
            sentence_log_prob = sum(log_token_probs) / len(log_token_probs) if len(log_token_probs) > 0 else None
            
            solutions.append(response)
            probabilities.append(sentence_log_prob)
        
        batch_solutions.append(solutions)
        batch_probabilities.append(probabilities)
    print(f'finished processing {len(input_texts)} inputs')
    return batch_solutions, batch_probabilities, log_all_token_probs

RuntimeError: Failed to import transformers.models.llama.modeling_llama because of the following error (look up to see its traceback):
/home/byuan48/anaconda3/envs/ReST/lib/python3.9/site-packages/flash_attn_2_cuda.cpython-39-x86_64-linux-gnu.so: undefined symbol: _ZN2at4_ops5zeros4callEN3c108ArrayRefINS2_6SymIntEEENS2_8optionalINS2_10ScalarTypeEEENS6_INS2_6LayoutEEENS6_INS2_6DeviceEEENS6_IbEE